In [43]:
from Bio import SeqIO
from Bio.Seq import Seq
import matplotlib.pyplot as plt
import seaborn as sns
from __future__ import annotations
import argparse
import json
import math
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Sequence
import numpy as np
import pandas as pd
from itertools import product



In [ ]:
AA20 = list("ACDEFGHIKLMNPQRSTVWY")  # canonical 20
AA20_SET = set(AA20)

In [51]:
CONFIG = {
    # Input files
    "COUNTS_PATH": "/Volumes/Elements/SerineProteaseLib_Quality/Results/Final_df.csv",          # Path to CSV/TSV with columns: sequence, count
    "INTENDED_PATH": None,                 # None or path to intended per-position CSV (position, aa, prob)

    # Number of samples
    "N_SAMPLES": 3,                     # Number of bootstrap samples for resampling
      # Intended distribution sources (used only if INTENDED_PATH is None)
    # Priority order: AA_PATTERN > INTENDED_FROM (codon patterns) > uniform
    #   AA_PATTERN: peptide-level pattern using literals, '.', and bracket sets.
    #       Example: "ACPA..[QY]...C"  (fixed ACPA, any AA, any AA, {Q,Y}, any, any, any, fixed C)
    "AA_PATTERN": "ACPA..[QY]...C",      # Set to None to disable AA pattern

    #   INTENDED_FROM: codon degeneracy (equal base mix), either a single pattern for all variable sites
    #   (e.g., "NNK"), or a list of length L with per-position codon patterns (e.g., ["ATG"] + ["NNK"]*(L-1)).
    "INTENDED_FROM": None,                # e.g., "NNK", "NNS", or ["ATG"] + ["NNK"]*(L-1)

    # Design
    # LENGTH is used when AA_PATTERN is None. If AA_PATTERN is provided, length is inferred from the pattern.
    "LENGTH": 11,
    "ALLOWED": "".join(AA20),             # String of allowed amino acids (no '*')

    # Scoring
    "WEIGHTS": (0.4, 0.4, 0.2),            # (w_div, w_uni, w_fid)
    "PENALTIES": (0.0, 2.0, 1.0),          # (alpha for stop, beta for indel, gamma for forbidden)

    # Diversity normalization
    "NMAX": None,                          # Optional override for theoretical design size; None to infer

    # Bootstrap
    "BOOTSTRAP": 0,                        # Number of bootstrap replicates for CIs (0 to skip)

    # Outputs
    "OUT_PREFIX": "qc_report"             # Prefix for output files
}

In [52]:
# Standard genetic code (DNA, T not U)
GENETIC_CODE = {
    "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
    "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
    "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
    "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
    "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G"
}

# IUPAC degenerate base codes (DNA), it maps each symbol to the set of nucleotides it represents.
IUPAC = {
    "N": "ACGT",
    "K": "GT",
    "S": "GC",
    "R": "AG",
    "M": "AC",
    "W": "AT",
    "Y": "CT",
    "H": "ACT",
    "B": "CGT",
    "V": "ACG",
    "D": "AGT",
    "A": "A",
    "C": "C",
    "G": "G",
    "T": "T",
}


In [53]:

def codons_from_pattern(pattern: str) -> List[str]:
    pattern = pattern.upper()
    if len(pattern) != 3:
        raise ValueError(f"Codon pattern must have length 3, got '{pattern}'")
    try:
        bases = [IUPAC[ch] for ch in pattern]
    except KeyError as e:
        raise ValueError(f"Unsupported IUPAC code in pattern '{pattern}': {e}")
    return ["".join(p) for p in product(*bases)]


def aa_distribution_from_pattern(pattern: str, allowed: Sequence[str], exclude_stop: bool = True) -> Tuple[Dict[str,float], float]:
    """Return (aa_prob_dict, stop_prob) from a codon degeneracy pattern.
    Assumes equal nucleotide mixture within each IUPAC set.
    If exclude_stop=True, AA probabilities are renormalized to sum to 1 over allowed AAs.
    """
    codons = codons_from_pattern(pattern)
    total = float(len(codons))
    counts: Dict[str, int] = {}
    for c in codons:
        aa = GENETIC_CODE[c]
        counts[aa] = counts.get(aa, 0) + 1
    stop_count = counts.get("*", 0)
    stop_prob = stop_count / total

    # Raw probabilities per AA (including stop)
    raw_probs = {aa: counts.get(aa, 0) / total for aa in set(GENETIC_CODE.values())}

    # Filter to allowed, exclude stop, and renormalize
    aa_probs = {a: raw_probs.get(a, 0.0) for a in allowed}
    if exclude_stop:
        s = sum(aa_probs.values())
        if s > 0:
            aa_probs = {a: p / s for a, p in aa_probs.items()}
    return aa_probs, stop_prob


def build_intended_from_patterns(patterns: Sequence[str], L: int, allowed: List[str]) -> Tuple[List[Dict[str,float]], List[float]]:
    """Build per-position intended AA distributions from codon patterns.
    patterns: length-1 (replicated) or length-L list of codon patterns.
    Returns (intended_list, stop_probs_per_position).
    """
    if len(patterns) == 1:
        patterns = list(patterns) * L
    if len(patterns) != L:
        raise ValueError(f"patterns length {len(patterns)} != L={L}")

    intended: List[Dict[str, float]] = []
    stop_probs: List[float] = []
    for pat in patterns:
        aa_probs, stop_p = aa_distribution_from_pattern(pat, allowed, exclude_stop=True)
        intended.append(aa_probs)
        stop_probs.append(stop_p)
    return intended, stop_probs


# -----------------------------
# AA-pattern (peptide-level) builder
# -----------------------------

def _strip_python_raw_string_literal(s: str) -> str:
    s = s.strip()
    if (s.startswith("r'") and s.endswith("'")) or (s.startswith('r"') and s.endswith('"')):
        s = s[2:-1]
    elif (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1]
    return s


def build_intended_from_aa_pattern(pattern: str, allowed: List[str]) -> List[Dict[str, float]]:
    """Parse a simple AA regex-like pattern into a per-position intended distribution.
    Supported tokens:
      - Single-letter AAs (e.g., 'A') → delta at that AA
      - '.' → any of the allowed AAs (uniform over `allowed`)
      - '[XYZ...]' → uniform over the listed AAs (must be subset of `allowed`)
    Example: "ACPA..[QY]...C" → length 11.
    """
    pat = _strip_python_raw_string_literal(str(pattern)).upper()
    intended: List[Dict[str, float]] = []
    i = 0
    allowed_set = set(allowed)
    while i < len(pat):
        ch = pat[i]
        if ch == '.':
            u = 1.0 / len(allowed)
            intended.append({a: u for a in allowed})
            i += 1
        elif ch == '[':
            j = pat.find(']', i + 1)
            if j == -1:
                raise ValueError("Unmatched '[' in AA pattern")
            opts = list(dict.fromkeys(pat[i + 1:j]))  # unique, preserve order
            if not opts:
                raise ValueError("Empty [] in AA pattern")
            for aa in opts:
                if aa not in allowed_set:
                    raise ValueError(f"AA '{aa}' in [] not in allowed set {allowed}")
            p = 1.0 / len(opts)
            intended.append({a: (p if a in opts else 0.0) for a in allowed})
            i = j + 1
        else:
            if ch not in allowed_set:
                raise ValueError(f"AA '{ch}' not in allowed set {allowed}")
            intended.append({a: (1.0 if a == ch else 0.0) for a in allowed})
            i += 1
    return intended


In [54]:
@dataclass
class Design:
    L: int
    allowed: List[str]  # allowed amino acids (global set)
    intended: List[Dict[str, float]]  # length L; per-position aa->prob (must sum to 1)


# -----------------------------
# Utility functions
# -----------------------------

def add_total_count(df: pd.DataFrame) -> pd.DataFrame:
    df["count"] = df.iloc[:, 1:CONFIG["N_SAMPLES"]].sum(axis=1)

    return df

def _safe_log(x: np.ndarray) -> np.ndarray:
    out = np.zeros_like(x, dtype=float)
    mask = x > 0
    out[mask] = np.log(x[mask])
    return out


def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen–Shannon divergence in nats for two discrete distributions over same support.
    p, q: 1D arrays summing to 1; zeros allowed.
    Returns JSD >= 0.
    """
    m = 0.5 * (p + q)
    # KL(p||m) with 0*log(0/x) = 0 by convention
    kl_pm = np.sum(np.where(p > 0, p * (_safe_log(p) - _safe_log(m)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * (_safe_log(q) - _safe_log(m)), 0.0))
    return 0.5 * (kl_pm + kl_qm)


def shannon_entropy(probs: np.ndarray) -> float:
    # H = -sum p log p with 0 log 0 := 0
    return -float(np.sum(np.where(probs > 0, probs * _safe_log(probs), 0.0)))


def normalize(v: np.ndarray) -> np.ndarray:
    s = v.sum()
    if s <= 0:
        return v
    return v / s


def infer_delimiter(path: str) -> str:
    if path.lower().endswith(".tsv"):
        return "\t"
    return ","


In [55]:
# -----------------------------
# Input parsing
# -----------------------------

def load_counts_table(path: str) -> pd.DataFrame:
    delim = infer_delimiter(path)
    df = pd.read_csv(path, sep=delim)
    df = add_total_count(df)
    cols = {c.lower().strip(): c for c in df.columns}
    seq_col = None
    for cand in ["sequence", "seq", "aa", "peptide"]:
        if cand in cols:
            seq_col = cols[cand]
            break
    if seq_col is None:
        raise ValueError("Counts file must contain a 'sequence' (or 'seq'/'aa'/'peptide') column")
    count_col = None
    for cand in ["count", "reads", "n", "c"]:
        if cand in cols:
            count_col = cols[cand]
            break
    if count_col is None:
        raise ValueError("Counts file must contain a 'count' (or 'reads') column")

    df = df[[seq_col, count_col]].copy()
    df.columns = ["sequence", "count"]
    df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
    df["count"] = df["count"].astype(int)
    df = df[df["count"] > 0]
    df = df.groupby("sequence", as_index=False)["count"].sum()
    return df


def load_intended(path: str, L: int, allowed: List[str]) -> List[Dict[str, float]]:
    delim = infer_delimiter(path)
    raw = pd.read_csv(path, sep=delim)
    req = {"position", "aa", "prob"}
    if not req.issubset(set(map(str.lower, raw.columns))):
        cols = {c.lower(): c for c in raw.columns}
        missing = req - set(cols)
        if missing:
            raise ValueError(f"Intended file missing required columns: {missing}")
        raw = raw.rename(columns={cols["position"]: "position", cols["aa"]: "aa", cols["prob"]: "prob"})
    else:
        raw = raw.rename(columns={"Position": "position", "AA": "aa", "Prob": "prob"}, errors="ignore")
    raw["position"] = raw["position"].astype(int)
    raw["aa"] = raw["aa"].astype(str).str.upper()
    raw["prob"] = raw["prob"].astype(float)

    intended = [{a: 0.0 for a in allowed} for _ in range(L)]
    for _, row in raw.iterrows():
        pos = int(row["position"]) - 1
        aa = row["aa"]
        p = float(row["prob"])
        if not (0 <= pos < L):
            continue
        if aa not in allowed:
            continue
        intended[pos][aa] = p
    for i in range(L):
        total = sum(intended[i].values())
        if total <= 0:
            u = 1.0 / len(allowed)
            intended[i] = {a: u for a in allowed}
        else:
            intended[i] = {a: intended[i][a] / total for a in allowed}
    return intended


def make_uniform_intended(L: int, allowed: List[str]) -> List[Dict[str, float]]:
    u = 1.0 / len(allowed)
    return [{a: u for a in allowed} for _ in range(L)]


In [56]:


# -----------------------------
# Core analysis
# -----------------------------

def analyze(df: pd.DataFrame, design: Design, alpha: float, beta: float, gamma: float,
            nmax: float | None, weights: Tuple[float, float, float]) -> Tuple[dict, dict, np.ndarray]:
    L = design.L
    allowed = set(design.allowed)

    total_reads = int(df["count"].sum())
    is_len_bad = df["sequence"].str.len() != L
    has_stop = df["sequence"].str.contains(r"\*")
    has_forbidden = ~df["sequence"].apply(lambda s: set(s).issubset(allowed))

    r_indel = float(df.loc[is_len_bad, "count"].sum()) / total_reads if total_reads else 0.0
    r_stop = float(df.loc[has_stop & ~is_len_bad, "count"].sum()) / total_reads if total_reads else 0.0
    r_forbid = float(df.loc[has_forbidden & ~is_len_bad & ~has_stop, "count"].sum()) / total_reads if total_reads else 0.0

    on_target_mask = (~is_len_bad) & (~has_stop) & (~has_forbidden)
    df_on = df.loc[on_target_mask].copy()

    on_target_reads = int(df_on["count"].sum())
    on_target_frac = on_target_reads / total_reads if total_reads else 0.0

    if on_target_reads > 0:
        q = df_on["count"].to_numpy(dtype=float) / float(on_target_reads)
    else:
        q = np.array([], dtype=float)

    H = shannon_entropy(q) if q.size else 0.0
    N_eff = float(math.exp(H))

    if nmax is None:
        nz_counts = [sum(1 for a, p in design.intended[i].items() if p > 0) for i in range(L)]
        n_design = 1.0
        for k in nz_counts:
            n_design *= max(1, k)
        nmax = n_design

    if nmax and nmax > 1:
        div_score = min(1.0, math.log(max(1.0, N_eff)) / math.log(nmax))
    else:
        div_score = 0.0

    alphabet = design.allowed
    aa_index = {a: i for i, a in enumerate(alphabet)}

    pos_counts = np.zeros((L, len(alphabet)), dtype=float)
    for _, row in df_on.iterrows():
        s = row["sequence"]
        c = int(row["count"])
        for i, ch in enumerate(s):
            pos_counts[i, aa_index[ch]] += c
    pos_totals = pos_counts.sum(axis=1, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        pos_freqs = np.divide(pos_counts, pos_totals, out=np.zeros_like(pos_counts), where=pos_totals>0)

    intended_mat = np.array([[design.intended[i][a] for a in alphabet] for i in range(L)], dtype=float)

    jsd = np.array([js_divergence(intended_mat[i], pos_freqs[i]) for i in range(L)])
    uni_pos = 1.0 - (jsd / math.log(2))
    uni_pos = np.nan_to_num(uni_pos, nan=0.0, posinf=0.0, neginf=0.0)
    uni_score = float(np.mean(uni_pos)) if L > 0 else 0.0

    fid_score = max(0.0, 1.0 - alpha * r_stop - beta * r_indel - gamma * r_forbid)

    w_div, w_uni, w_fid = weights
    qc = w_div * div_score + w_uni * uni_score + w_fid * fid_score

    per_pos = {
        "position": list(range(1, L + 1)),
        "uni": uni_pos.tolist(),
        "jsd_nats": jsd.tolist(),
        "jsd_bits": (jsd / math.log(2)).tolist(),
        "observed_total_reads": pos_totals.flatten().astype(int).tolist(),
    }

    freqs_tables = []
    for i in range(L):
        df_i = pd.DataFrame({
            "position": i + 1,
            "aa": alphabet,
            "freq": pos_freqs[i].tolist(),
            "count": pos_counts[i].astype(int).tolist(),
        })
        freqs_tables.append(df_i)

    summary = {
        "length": L,
        "total_reads": total_reads,
        "on_target_reads": on_target_reads,
        "on_target_fraction": on_target_frac,
        "error_rates": {
            "stop": r_stop,
            "indel": r_indel,
            "forbidden": r_forbid,
        },
        "entropy": H,
        "effective_diversity": N_eff,
        "nmax": nmax,
        "DivScore": div_score,
        "UniScore": uni_score,
        "FidScore": fid_score,
        "weights": {
            "div": w_div,
            "uni": w_uni,
            "fid": w_fid,
        },
        "QC": qc,
    }

    return summary, per_pos, freqs_tables


# -----------------------------
# Bootstrap
# -----------------------------

def bootstrap_metrics(df: pd.DataFrame, design: Design, alpha: float, beta: float, gamma: float,
                      nmax: float | None, weights: Tuple[float, float, float], B: int,
                      rng: np.random.Generator | None = None) -> dict:
    if B <= 0:
        return {}
    if rng is None:
        rng = np.random.default_rng()

    allowed = set(design.allowed)
    L = design.L

    is_len_bad = df["sequence"].str.len() != L
    has_stop = df["sequence"].str.contains(r"\*")
    has_forbidden = ~df["sequence"].apply(lambda s: set(s).issubset(allowed))
    on_target_mask = (~is_len_bad) & (~has_stop) & (~has_forbidden)
    df_on = df.loc[on_target_mask].copy()

    total_reads = int(df["count"].sum())
    if total_reads == 0:
        return {}

    counts = df_on["count"].to_numpy(dtype=int)
    if counts.size == 0:
        return {}

    probs = counts / counts.sum()
    seqs = df_on["sequence"].tolist()

    alphabet = design.allowed
    aa_index = {a: i for i, a in enumerate(alphabet)}
    intended_mat = np.array([[design.intended[i][a] for a in alphabet] for i in range(L)], dtype=float)

    def metrics_from_sampled_counts(sample_counts: np.ndarray) -> Tuple[float, float]:
        on_reads = int(sample_counts.sum())
        q = sample_counts / on_reads if on_reads > 0 else np.array([], dtype=float)
        H = shannon_entropy(q) if on_reads > 0 else 0.0
        N_eff = float(math.exp(H))
        if nmax and nmax > 1:
            div_score = min(1.0, math.log(max(1.0, N_eff)) / math.log(nmax))
        else:
            div_score = 0.0
        pos_counts = np.zeros((L, len(alphabet)), dtype=float)
        for s, c in zip(seqs, sample_counts):
            for i, ch in enumerate(s):
                pos_counts[i, aa_index[ch]] += c
        pos_totals = pos_counts.sum(axis=1, keepdims=True)
        with np.errstate(invalid='ignore', divide='ignore'):
            pos_freqs = np.divide(pos_counts, pos_totals, out=np.zeros_like(pos_counts), where=pos_totals>0)
        jsd = np.array([js_divergence(intended_mat[i], pos_freqs[i]) for i in range(L)])
        uni_score = float(np.mean(1.0 - (jsd / math.log(2)))) if L > 0 else 0.0
        return div_score, uni_score

    r_indel = float(df.loc[is_len_bad, "count"].sum()) / total_reads
    r_stop = float(df.loc[has_stop & ~is_len_bad, "count"].sum()) / total_reads
    r_forbid = float(df.loc[has_forbidden & ~is_len_bad & ~has_stop, "count"].sum()) / total_reads
    fid_score = max(0.0, 1.0 - alpha * r_stop - beta * r_indel - gamma * r_forbid)

    w_div, w_uni, w_fid = weights

    divs = np.zeros(B)
    unis = np.zeros(B)
    qcs = np.zeros(B)

    on_reads = counts.sum()

    for b in range(B):
        sample_counts = rng.multinomial(on_reads, probs)
        div_b, uni_b = metrics_from_sampled_counts(sample_counts)
        divs[b] = div_b
        unis[b] = uni_b
        qcs[b] = w_div * div_b + w_uni * uni_b + w_fid * fid_score

    def ci(arr: np.ndarray) -> Tuple[float, float]:
        return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))

    out = {
        "DivScore_CI95": ci(divs),
        "UniScore_CI95": ci(unis),
        "QC_CI95": ci(qcs),
    }
    return out


# -----------------------------
# Helpers
# -----------------------------

def estimate_design_size(intended: List[Dict[str, float]]) -> int:
    n = 1
    for pos in intended:
        k = sum(1 for p in pos.values() if p > 0)
        n *= max(1, k)
    return int(n)


# -----------------------------
# Runner (uses CONFIG)
# -----------------------------
if __name__ == "__main__":
    COUNTS_PATH = CONFIG["COUNTS_PATH"]
    INTENDED_PATH = CONFIG["INTENDED_PATH"]
    AA_PATTERN = CONFIG.get("AA_PATTERN", None)
    INTENDED_FROM = CONFIG.get("INTENDED_FROM", None)
    ALLOWED = [c for c in str(CONFIG["ALLOWED"]).upper() if c.isalpha() and c != 'X'] or AA20

    if not os.path.exists(COUNTS_PATH):
        raise FileNotFoundError(f"Counts file not found: {COUNTS_PATH}")

    # Build intended + determine L
    intended_stop_probs: Optional[List[float]] = None 
    intended: List[Dict[str, float]]

    if INTENDED_PATH:
        L = int(CONFIG["LENGTH"])
        intended = load_intended(INTENDED_PATH, L, ALLOWED)
    elif AA_PATTERN:
        intended = build_intended_from_aa_pattern(AA_PATTERN, ALLOWED)
        L = len(intended)
    elif INTENDED_FROM:
        L = int(CONFIG["LENGTH"])
        if isinstance(INTENDED_FROM, str):
            intended, intended_stop_probs = build_intended_from_patterns([INTENDED_FROM.upper()], L, ALLOWED)
        elif isinstance(INTENDED_FROM, (list, tuple)):
            intended, intended_stop_probs = build_intended_from_patterns(list(INTENDED_FROM), L, ALLOWED)
        else:
            raise ValueError(f"Unsupported INTENDED_FROM: {INTENDED_FROM}")
    else:
        L = int(CONFIG["LENGTH"])
        intended = make_uniform_intended(L, ALLOWED)

    # Load counts AFTER deciding L (for length-based filtering checks later)
    df = load_counts_table(COUNTS_PATH)

    # NMAX override
    NMAX = CONFIG["NMAX"]

    design = Design(L=L, allowed=ALLOWED, intended=intended)

    summary, per_pos, freqs_tables = analyze(
        df=df,
        design=design,
        alpha=CONFIG["PENALTIES"][0],
        beta=CONFIG["PENALTIES"][1],
        gamma=CONFIG["PENALTIES"][2],
        nmax=NMAX,
        weights=tuple(CONFIG["WEIGHTS"]),
    )

    # Attach intended stop probabilities (if we used codon patterns)
    if intended_stop_probs is not None:
        summary["intended_stop_probability_per_position"] = intended_stop_probs

    boot = bootstrap_metrics(
        df=df,
        design=design,
        alpha=CONFIG["PENALTIES"][0],
        beta=CONFIG["PENALTIES"][1],
        gamma=CONFIG["PENALTIES"][2],
        nmax=NMAX,
        weights=tuple(CONFIG["WEIGHTS"]),
        B=int(CONFIG["BOOTSTRAP"]),
    )
    if boot:
        summary.update(boot)

    os.makedirs(os.path.dirname(CONFIG["OUT_PREFIX"]) or ".", exist_ok=True)

    with open(f"{CONFIG["OUT_PREFIX"]}.summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    per_pos_df = pd.DataFrame(per_pos)
    per_pos_df.to_csv(f"{CONFIG["OUT_PREFIX"]}.per_position.csv", index=False)

    for i, df_i in enumerate(freqs_tables, start=1):
        df_i.to_csv(f"{CONFIG["OUT_PREFIX"]}.freqs_pos{i}.csv", index=False)

    print(json.dumps(summary, indent=2))


{
  "length": 11,
  "total_reads": 21370524,
  "on_target_reads": 17910945,
  "on_target_fraction": 0.8381144514753125,
  "error_rates": {
    "stop": 0.09775909097970643,
    "indel": 0.06412636395813223,
    "forbidden": 9.358684887651796e-08
  },
  "entropy": 12.442290527168229,
  "effective_diversity": 253290.04327119942,
  "nmax": 6400000.0,
  "DivScore": 0.7939281856843617,
  "UniScore": 0.9435650501294858,
  "FidScore": 0.8717471784968868,
  "weights": {
    "div": 0.4,
    "uni": 0.4,
    "fid": 0.2
  },
  "QC": 0.8693467300249164
}
